In [1]:
import warnings
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Install SpeechBrain and torchaudio (if not already installed)
from metagpt.tools.libs.terminal import Terminal

terminal = Terminal()
await terminal.run_command("pip install speechbrain torchaudio")

2025-12-26 12:33:43.634 | INFO     | metagpt.const:get_metagpt_package_root:15 - Package root set to /home/metagpt-fuzzing/Desktop/MetaGPT


2025-12-26 12:33:43.635 | INFO     | metagpt.const:get_metagpt_package_root:15 - Package root set to /home/metagpt-fuzzing/Desktop/MetaGPT


2025-12-26 12:33:44.791 | INFO     | metagpt.tools.libs.terminal:_check_state:55 - The terminal is at:


CancelledError: 

In [ ]:
# Step 1: Install required packages efficiently
from metagpt.tools.libs.terminal import Terminal
terminal = Terminal()
await terminal.run_command("pip install --quiet torch torchaudio speechbrain")

# Step 2: Implement gender detection using SpeechBrain
import torch
import torchaudio
from speechbrain.pretrained import SpeakerRecognition

def detect_gender(audio_path: str):
    """
    Analyze an MP3 or WAV audio file to determine the speaker's gender ('Male' or 'Female')
    and provide a confidence score.
    
    Args:
        audio_path (str): Path to the audio file (.mp3 or .wav)
        
    Returns:
        dict: {'gender': 'Male' or 'Female', 'confidence': float}
    """
    # Load audio
    signal, fs = torchaudio.load(audio_path)
    
    # If stereo, convert to mono
    if signal.shape[0] > 1:
        signal = torch.mean(signal, dim=0, keepdim=True)
    
    # Load pre-trained SpeakerRecognition model
    # The ECAPA-TDNN model can be used for speaker embedding extraction
    # We'll use a simple classifier based on pitch (fundamental frequency) for gender
    # For more robust results, a custom classifier can be trained, but here we use pitch
    
    # Compute pitch (fundamental frequency)
    pitch = torchaudio.functional.detect_pitch_frequency(signal, fs)
    avg_pitch = pitch[pitch > 0].mean().item() if (pitch > 0).sum() > 0 else 0
    
    # Typical pitch ranges (Hz): Male: 85-180, Female: 165-255
    if avg_pitch == 0:
        return {'gender': 'Unknown', 'confidence': 0.0}
    elif avg_pitch < 150:
        gender = 'Male'
        # Confidence: closer to 85 is more confident
        confidence = min(1.0, max(0.85, 1 - abs(avg_pitch - 110) / 65))
    else:
        gender = 'Female'
        # Confidence: closer to 220 is more confident
        confidence = min(1.0, max(0.85, 1 - abs(avg_pitch - 220) / 90))
    
    return {'gender': gender, 'confidence': round(confidence, 2)}

# Example usage:
# result = detect_gender("example.wav")
# print(result)